In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\YASHUB
[nltk_data]     HAYAT\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\YASHUB
[nltk_data]     HAYAT\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score

In [5]:
import pickle

In [14]:
data = pd.read_csv(r"D:\data_science_4achievers\dataset_project2\Resume.csv")

In [7]:
with open("tfidf2.pkl", "rb") as f2:
    loaded_tfidf2 = pickle.load(f2)

with open("job_desc_vectors.pkl", "rb") as f2:
    loaded_job_desc_vectors = pickle.load(f2)

with open("job_roles.pkl", "rb") as f2:
    loaded_job_roles = pickle.load(f2)

print("All files loaded successfully!")
print("Total job roles loaded:", len(loaded_job_roles))

All files loaded successfully!
Total job roles loaded: 24


In [10]:
with open('resume_classifier.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('tfidf_vectorizer.pkl', 'rb') as f:
    loaded_tfidf = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    loaded_le = pickle.load(f)

print("All files loaded successfully!")

All files loaded successfully!


In [16]:
def clean_resume(text):

    text = str(text)

    # remove links
    text = re.sub(r'http\S+|www\S+|linkedin\S+|github\S+', ' ', text)

    # remove dates like Jan 2015, March 2012
    text = re.sub(
        r'\b(jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|september|oct|october|nov|november|dec|december)\s+\d{4}\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove extra spaces/newlines
    text = re.sub(r'\s+', ' ', text)

    # lowercase
    text = text.lower().strip()

    return text

data["Resume_clean"] = data["Resume_str"].apply(clean_resume) 

In [18]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def lemmatize_text(text):
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

data["Resume_clean"] = data["Resume_clean"].apply(lemmatize_text)

In [ ]:
def predict_job_category(resume_text):
    cleaned = clean_resume(resume_text)
    lemmatized = lemmatize_text(cleaned)
    
    vectorized = loaded_tfidf.transform([lemmatized])
    vectorized_df = pd.DataFrame(
        vectorized.toarray(),
        columns=loaded_tfidf.get_feature_names_out()
    )
    prediction = loaded_model.predict(vectorized_df)
    category = loaded_le.inverse_transform(prediction)[0]
    return category
    

def predict_function_top3(res_index):
    resume_text1 = data["Resume_clean"].iloc[res_index]
    cleaned2 = clean_resume(resume_text1)
    lemmatized = lemmatize_text(cleaned2)
    vectorized = loaded_tfidf2.transform([lemmatized])
    similarity_scores = cosine_similarity(
        vectorized,
        loaded_job_desc_vectors
    ).flatten()
    top3_indices1 = similarity_scores.argsort()[::-1][:3]
    print(f"\nResume{res_index} - Actual Category: {data["Category"].iloc[res_index]}")
    print("Top 3 Matching Roles:")
    print("-" * 40)
    for rank, ind in enumerate(top3_indices1, 1):
        print(f"{rank}. {loaded_job_roles[ind]} -> Score: {similarity_scores[ind]:.4f}")


In [22]:
def combined_pred_func(resume_idx):
    resume_text2 = data["Resume_clean"].iloc[resume_idx]
    cleaned3 = clean_resume(resume_text2)
    lemmatized2 = lemmatize_text(cleaned3)
    
    vectorized_top3 = loaded_tfidf2.transform([lemmatized2])
    vectorized_classify = loaded_tfidf.transform([lemmatized2])
    vectorized_classify = pd.DataFrame(
        vectorized_classify.toarray(),
        columns=loaded_tfidf.get_feature_names_out()
    )
    
    similarity_scores1 = cosine_similarity(
        vectorized_top3,
        loaded_job_desc_vectors
    ).flatten()
    
    top3_indices1 = similarity_scores1.argsort()[::-1][:3]
    
    prediction = loaded_model.predict(vectorized_classify)
    category = loaded_le.inverse_transform(prediction)[0]
    
    print("="*40)
    print("RESUME SCREENING RESULTS")
    print("="*40)
    print(f"Actual Category    : {data['Category'].iloc[resume_idx]}")
    print(f"Predicted Category : {category}")
    print("\nTop 3 Matching Roles:")
    print("-"*40)
    for rank, ind in enumerate(top3_indices1, 1):
        print(f"{rank}. {loaded_job_roles[ind]} -> Score: {similarity_scores1[ind]:.4f}")
    print("="*40)
    
    return category


for i in [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700]:
    combined_pred_func(i)

    
    
    
    
    
    

RESUME SCREENING RESULTS
Actual Category    : HR
Predicted Category : HR

Top 3 Matching Roles:
----------------------------------------
1. HR -> Score: 0.1556
2. ADVOCATE -> Score: 0.1247
3. DIGITAL-MEDIA -> Score: 0.1020
RESUME SCREENING RESULTS
Actual Category    : HR
Predicted Category : HR

Top 3 Matching Roles:
----------------------------------------
1. HR -> Score: 0.1813
2. ACCOUNTANT -> Score: 0.0816
3. ADVOCATE -> Score: 0.0699
RESUME SCREENING RESULTS
Actual Category    : HR
Predicted Category : HR

Top 3 Matching Roles:
----------------------------------------
1. HR -> Score: 0.0911
2. ADVOCATE -> Score: 0.0767
3. HEALTHCARE -> Score: 0.0647
RESUME SCREENING RESULTS
Actual Category    : DESIGNER
Predicted Category : DESIGNER

Top 3 Matching Roles:
----------------------------------------
1. DESIGNER -> Score: 0.1570
2. CONSULTANT -> Score: 0.0869
3. CONSTRUCTION -> Score: 0.0867
RESUME SCREENING RESULTS
Actual Category    : DESIGNER
Predicted Category : DESIGNER

Top 3 Mat

In [24]:
import os
print(os.getcwd())

C:\Users\YASHUB HAYAT\Resume_project
